# Healthcare Analytics EDA

In [ ]:
import numpy as np
import pandas as pd

In [12]:
# 1. Load the raw dataset
from google.colab import files
uploaded=files.upload()

df = pd.read_csv('Hospital ER_Data.csv')

Saving Hospital ER_Data.csv to Hospital ER_Data (1).csv


In [13]:
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9216 entries, 0 to 9215
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Patient Id                  9216 non-null   object 
 1   Patient Admission Date      9216 non-null   object 
 2   Patient First Inital        9216 non-null   object 
 3   Patient Last Name           9216 non-null   object 
 4   Patient Gender              9216 non-null   object 
 5   Patient Age                 9216 non-null   int64  
 6   Patient Race                9216 non-null   object 
 7   Department Referral         3816 non-null   object 
 8   Patient Admission Flag      9216 non-null   bool   
 9   Patient Satisfaction Score  2517 non-null   float64
 10  Patient Waittime            9216 non-null   int64  
 11  Patients CM                 9216 non-null   int64  
dtypes: bool(1), float64(1), int64(3), object(7)
memory usage: 801.1+ KB


,Patient Id,Patient Admission Date,Patient First Inital,Patient Last Name,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime,Patients CM
count,9216,9216,9216,9216,9216,9216.000000,9216,3816,9216,2517.000000,9216.000000,9216.000000
unique,9216,9176,26,8400,3,NaN,7,7,2,NaN,NaN,NaN
top,135-96-4722,19-12-2023 20:24,C,Kilduff,M,NaN,White,General Practice,True,NaN,NaN,NaN
freq,1,2,383,4,4705,NaN,2571,1840,4612,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,39.855143,NaN,NaN,NaN,4.992054,35.259874,0.052083
std,NaN,NaN,NaN,NaN,NaN,22.755125,NaN,NaN,NaN,3.138043,14.735323,0.222207
min,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,0.000000,10.000000,0.000000
25%,NaN,NaN,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,2.000000,23.000000,0.000000
50%,NaN,NaN,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,5.000000,35.000000,0.000000
75%,NaN,NaN,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,8.000000,48.000000,0.000000


In [14]:
# Convert admission date to datetime object
df['Patient Admission Date'] = pd.to_datetime(
    df['Patient Admission Date'], format='%d-%m-%Y %H:%M'
)

In [15]:
# 1. Admission Hour (Extract hour 0-23 from timestamp)
df['Admission Hour'] = df['Patient Admission Date'].dt.hour


In [16]:
# 2. Admission Status (Convert True/False flag to 'Admitted'/'Not Admitted')
df['Admission status'] = np.where(
    df['Patient Admission Flag'] == True, 'Admitted', 'Not Admitted'
)

In [17]:
# 3. Age Group (Binned age ranges matching DAX SWITCH logic)
def get_age_group(age):
  if age >= 100:
    return '100+'
  elif age >= 90:
    return '90-99'
  elif age >= 80:
    return '80-89'
  elif age >= 70:
    return '70-79'
  elif age >= 60:
    return '60-69'
  elif age >= 50:
    return '50-59'
  elif age >= 40:
    return '40-49'
  elif age >= 30:
    return '30-39'
  elif age >= 20:
    return '20-29'
  elif age >= 10:
    return '10-19'
  else:
    return '0-9'


df['Age group'] = df['Patient Age'].apply(get_age_group)

In [18]:
def get_waittime_interval(hour):
  if hour < 2:
    return '00-02'
  elif hour < 4:
    return '03-04'
  elif hour < 6:
    return '05-06'
  elif hour < 8:
    return '07-08'
  elif hour < 10:
    return '09-10'
  elif hour < 12:
    return '11-12'
  elif hour < 14:
    return '13-14'
  elif hour < 16:
    return '15-16'
  elif hour < 18:
    return '17-18'
  elif hour < 20:
    return '19-20'
  elif hour < 22:
    return '21-22'
  elif hour < 24:
    return '23-24'
  else:
    return 'above 24'


df['Waittime interval'] = df['Admission Hour'].apply(get_waittime_interval)


In [20]:
# 5. Waittime Status (Target compliance check <= 30 mins)
df['Waittime status'] = np.where(
    df['Patient Waittime'] <= 30, 'Within Target', 'Target missed')

In [21]:
# 6. Patient Admin Date (Date without time component)
df['Patient Admin Date'] = df['Patient Admission Date'].dt.date

In [22]:
df.head()

,Patient Id,Patient Admission Date,Patient First Inital,Patient Last Name,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime,Patients CM,Admission Hour,Admission status,Age group,Waittime interval,Waittime status,Patient Admin Date
0,145-39-5406,2024-03-20 08:47:00,H,Glasspool,M,69,White,NaN,False,10.0,39,0,8,Not Admitted,60-69,09-10,Target missed,2024-03-20
1,316-34-3057,2024-06-15 11:29:00,X,Methuen,M,4,Native American/Alaska Native,NaN,True,NaN,27,0,11,Admitted,0-9,11-12,Within Target,2024-06-15
2,897-46-3852,2024-06-20 09:13:00,P,Schubuser,F,56,African American,General Practice,True,9.0,55,0,9,Admitted,50-59,09-10,Target missed,2024-06-20
3,358-31-9711,2024-02-04 22:34:00,U,Titcombe,F,24,Native American/Alaska Native,General Practice,True,8.0,31,0,22,Admitted,20-29,23-24,Target missed,2024-02-04
4,289-26-0537,2024-09-04 17:48:00,Y,Gionettitti,M,5,African American,Orthopedics,False,NaN,10,0,17,Not Admitted,0-9,17-18,Within Target,2024-09-04


In [23]:
df.to_csv('Hospital_ER_Data_Updated.csv', index=False)